<a href="https://colab.research.google.com/github/Darya-06/python-ai-Aleinik-Darya/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- `animated_television_series.csv` — информация об анимационных телесериалах (название, страна, композитор, рейтинг и производные работы)

**Что мы делаем:**
1. Клонируем ваш репозиторий GitHub в Colab
2. Читаем CSV-файл в pandas DataFrame
3. Очищаем и анализируем структуру столбцов
4. Выполняем быструю валидацию данных (проверка пропусков, уникальных значений)

## 🐱 [1] Клонируем репозиторий курса в Colab

In [1]:
# 🐱 Шаг 1. Клонируем ваш репозиторий в Colab

import os

if not os.path.exists("python-ai-Aleinik-Darya"):
    !git clone -q https://github.com/Darya-06/python-ai-Aleinik-Darya.git

%cd python-ai-Aleinik-Darya

print("✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Aleinik-Darya")

/content/python-ai-Aleinik-Darya
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Aleinik-Darya


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [17]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd

df_series = pd.read_csv("data/animated_television_series.csv")


print("✅ Загружено строк в df_series:", len(df_series))


✅ Загружено строк в df_series: 4557


## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `serial` с URL Wikidata (например, `http://www.wikidata.org/entity/Q77522`) — нам не нужна ссылка, достаточно читаемого названия сериала.
- Столбцы `serialLabel`, `countryLabel`, `composerLabel`, `rarsRatingLabel`, `derivativeWorkLabel` содержат человекочитаемые значения.

В этом шаге мы:
- удалим столбец `serial` с техническими URL;
- переименуем все столбцы с суффиксом `Label`, убрав его (`serialLabel → serial`, `countryLabel → country` и т.д.);
- получим чистую таблицу для дальнейшего анализа.

> ⚠️ **Важно:** столбец `rarsRating` может содержать текстовые значения (например, "18+"), поэтому числовое преобразование здесь не применяется.

In [25]:
# 🧹 Шаг 2B. Очистка и переименование столбцов (безопасно для повторного запуска)

# 1) Удаляем технический столбец с URL Wikidata, если он существует
if "serial" in df_series.columns and "serialLabel" in df_series.columns:
    # Удаляем ТОЛЬКО если есть и URL-столбец, и его человекочитаемая версия
    # (защита от случайного удаления уже очищенных данных)
    df_series = df_series.drop(columns=["serial"])
    print("🗑️ Удалён столбец 'serial' с URL Wikidata")
elif "serial" in df_series.columns and "serialLabel" not in df_series.columns:
    print("⚠️ Столбец 'serial' присутствует без 'serialLabel' — возможно, данные уже очищены")
else:
    print("ℹ️ Столбец 'serial' с URL уже удалён")

# 2) Автоматически убираем суффикс 'Label' у всех столбцов, которые его содержат
columns_to_rename = {
    col: col.replace("Label", "")
    for col in df_series.columns
    if col.endswith("Label")
}

if columns_to_rename:
    df_series = df_series.rename(columns=columns_to_rename)
    print(f"✏️ Убран суффикс 'Label' у столбцов: {list(columns_to_rename.keys())} → {list(columns_to_rename.values())}")
else:
    print("ℹ️ Суффикс 'Label' уже удалён из всех столбцов")

# 3) Итоговый вывод
print("\n✅ Данные очищены и готовы к анализу")
print("📋 Итоговая структура столбцов:", list(df_series.columns))
print("\n🔍 Пример данных после очистки:")
display(df_series.head(3))

⚠️ Столбец 'serial' присутствует без 'serialLabel' — возможно, данные уже очищены
ℹ️ Суффикс 'Label' уже удалён из всех столбцов

✅ Данные очищены и готовы к анализу
📋 Итоговая структура столбцов: ['serial', 'country', 'composer', 'rarsRating', 'derivativeWork']

🔍 Пример данных после очистки:


,serial,country,composer,rarsRating,derivativeWork
0,Клуб Винкс,Италия,Фабрицио Кастаниа,NaN,Клуб Винкс: Волшебное приключение
1,Клуб Винкс,Италия,Фабрицио Кастаниа,NaN,Судьба: Сага Винкс
2,Футурама,США,Кристофер Тинг,18+,NaN


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор обоих DataFrame:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по бюджету (`capital_cost`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы не повторять один и тот же код два раза.

In [ ]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

# 🔍 Шаг 3. Обзор данных

show_info(df_cost, "Бюджеты мультфильмов (df_cost)")
show_info(df_genre, "Жанры, страны и продолжительность (df_genre)")

print("\n📈 Статистика по бюджету (capital_cost):")
print(df_cost["capital_cost"].describe())


📊 Бюджеты мультфильмов (df_cost)
Размер: (421, 2)
Столбцы: film, capital_cost

Первые строки:
            film  capital_cost
0     Покахонтас      55000000
1   Трансформеры       6000000
2            Рио      90000000
3    Огонь и лёд       1200000
4  Цыплёнок Цыпа     150000000

📊 Жанры, страны и продолжительность (df_genre)
Размер: (8011, 4)
Столбцы: film, genre, country, duration

Первые строки:
                                              film                genre  \
0  Паутина Шарлотты 2: Великое приключение Уилбура    музыкальный фильм   
1  Паутина Шарлотты 2: Великое приключение Уилбура     комедийная драма   
2                           Рождественская история  драматический фильм   
3                           Рождественская история    фэнтезийный фильм   
4                           Рождественская история           мультфильм   

  country  duration  
0     США        79  
1     США        79  
2     США        96  
3     США        96  
4     США        96  

📈 Статистика 

In [ ]:
# То же самое, но в миллионах долларов (млн $)
stats_millions = (df_cost["capital_cost"] / 1e6).describe().round(2)
print("\n📈 Статистика по бюджету в млн $:")
print(stats_millions)


📈 Статистика по бюджету в млн $:
count     421.00
mean       70.66
std       165.69
min         0.00
25%         5.50
50%        30.00
75%        90.00
max      2800.00
Name: capital_cost, dtype: float64


## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** фильмов, стран и жанров есть в данных;
- **какие страны встречаются чаще всего** (Топ‑5 по числу строк);
- **какие жанры самые популярные** (Топ‑10 по числу строк).

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому  
`df_genre["country"].value_counts().head()` даёт **Топ‑5 стран по числу записей**.

In [ ]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных")

# Датасет 1: бюджеты
print("\nУникальных мультфильмов в df_cost:", df_cost["film"].nunique())
print("Диапазон бюджетов (млн $):",
      df_cost["capital_cost"].min() / 1e6, "—", df_cost["capital_cost"].max() / 1e6)

# Датасет 2: жанры, страны, длительность
print("\nУникальных мультфильмов в df_genre:", df_genre["film"].nunique())
print("Уникальных стран:", df_genre["country"].nunique())
print("Уникальных жанров:", df_genre["genre"].nunique())

print("\nТоп-5 стран по числу записей:")
print(df_genre["country"].value_counts().head())

print("\nТоп-10 жанров:")
print(df_genre["genre"].value_counts().head(10))

🔍 Быстрая проверка данных

Уникальных мультфильмов в df_cost: 417
Диапазон бюджетов (млн $): 9e-06 — 2800.0

Уникальных мультфильмов в df_genre: 2273
Уникальных стран: 86
Уникальных жанров: 240

Топ-5 стран по числу записей:
country
США        2836
Франция     527
СССР        488
Дания       338
Россия      334
Name: count, dtype: int64

Топ-10 жанров:
genre
приключенческий фильм          970
комедийный фильм               833
фэнтезийный фильм              778
семейный фильм                 735
детский фильм                  627
мультфильм                     456
музыкальный фильм              304
драматический фильм            256
боевик                         249
научно-фантастический фильм    220
Name: count, dtype: int64


## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Посмотрели базовую статистику по бюджету (`capital_cost`)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨